# Project Setup

## Overview
This notebook is part of a reproducible GRPO-based fine-tuning pipeline for cybersecurity policy generation.

## Purpose of this setup cell
The first code cell in this notebook:
- loads environment variables from `.env`
- identifies the project root directory automatically
- defines standard folder paths used across the project
- creates required folders if they do not already exist

## Why this matters
This makes the notebook portable across macOS, Windows, and Linux without requiring users to manually edit file paths.

## Expected project folders
- `data/corpus/` → source PDF corpus
- `data/processed/` → intermediate processed data
- `data/sample/` → optional small example data
- `outputs/completions/` → generated completions
- `outputs/rankings/` → ranked outputs
- `outputs/models/` → trained model checkpoints
- `outputs/evaluations/` → evaluation results

In [ ]:
import os
from pathlib import Path
from dotenv import load_dotenv

load_dotenv()

# Define PROJECT_ROOT automatically
if Path.cwd().name == "notebooks":
    PROJECT_ROOT = Path.cwd().parent
else:
    PROJECT_ROOT = Path.cwd()

# Define folders
DATA_DIR = PROJECT_ROOT / "data"
CORPUS_DIR = DATA_DIR / "corpus"
PROCESSED_DIR = DATA_DIR / "processed"

OUTPUT_DIR = PROJECT_ROOT / "outputs"
COMPLETIONS_DIR = OUTPUT_DIR / "completions"
RANKINGS_DIR = OUTPUT_DIR / "rankings"
MODELS_DIR = OUTPUT_DIR / "models"
EVAL_DIR = OUTPUT_DIR / "evaluations"

# Create folders if needed
for folder in [
    DATA_DIR, CORPUS_DIR, PROCESSED_DIR,
    OUTPUT_DIR, COMPLETIONS_DIR, RANKINGS_DIR, MODELS_DIR, EVAL_DIR
]:
    folder.mkdir(parents=True, exist_ok=True)

print("PROJECT_ROOT:", PROJECT_ROOT)

# Step 4 — Fine-Tune the Model with GRPO

## Purpose
This notebook performs GRPO-based fine-tuning using the ranked completion groups created in Step 3.

## Why this step is important
This is the training phase of the pipeline. It uses grouped preference information to improve the model so that it learns to generate better cybersecurity policy outputs.

## Inputs
- Ranked completion groups from `outputs/rankings/`
- Hugging Face token from `.env`

## Outputs
- Fine-tuned checkpoints saved to `outputs/models/`

## Main tasks
1. Load the ranked training data
2. Prepare the base model and tokenizer
3. Configure GRPO and LoRA settings
4. Run fine-tuning
5. Save training checkpoints and the best model

## Success criteria
This step is complete when the fine-tuned model checkpoints are successfully written to the model's output folder.

In [ ]:
# ================= GRPO Fine-Tuning with Metrics + Early Stopping + TQDM =================
# 
# pip install -U "transformers>=4.43,<5" "peft>=0.11.1,<0.13" "accelerate>=0.30.0" "safetensors>=0.4.2" "bitsandbytes>=0.43.0" tqdm

import os, json, math, random
from dataclasses import dataclass
from typing import List, Dict, Any, Optional

import torch
import torch.nn.functional as F


from torch.utils.data import Dataset, DataLoader
from tqdm.auto import tqdm, trange
from huggingface_hub import login

# ---- Import transformers module first, add shim for PEFT on Transformers v5+ ----
import transformers as _tfm
if not hasattr(_tfm, "PreTrainedModel"):
    try:
        from transformers.modeling_utils import PreTrainedModel as _PreTrainedModel
        _tfm.PreTrainedModel = _PreTrainedModel
        print("[shim] Attached transformers.PreTrainedModel for PEFT compatibility.")
    except Exception as e:
        raise ImportError(
            "Transformers v5+ detected and PEFT requires transformers.PreTrainedModel. "
            "Either pin transformers<5 / peft<0.13 or keep this shim."
        ) from e

from transformers import AutoTokenizer, AutoModelForCausalLM, get_cosine_schedule_with_warmup
from peft import LoraConfig, get_peft_model
# from bitsandbytes import load_and_colorize_empty_state_dict # Removed import

# -------------------------- CONFIG --------------------------
HF_TOKEN = os.getenv("HF_TOKEN")
login(HF_TOKEN)


MODEL_ID   = "meta-llama/Llama-3.2-1B-Instruct"
DATA_JSONL = "/content/cyber_policies_4comps-2_grpo_ranked.jsonl"
OUTPUT_DIR = "/content/grpo_lora_ckpt"

OUTPUT_DIR = "/content/drive/MyDrive/grpo_lora_ckpt"
BEST_DIR   = OUTPUT_DIR + "/best"

import os
os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(BEST_DIR, exist_ok=True)



# Train knobs
VAL_FRACTION       = 0.5
BATCH_SIZE         = 2 # Reduced batch size
GRAD_ACCUM_STEPS   = 4 # Increased gradient accumulation steps
NUM_EPOCHS         = 40
LR                 = 2e-4
WEIGHT_DECAY       = 0.05
WARMUP_STEPS       = 200
MAX_STEPS          = None
CLIP_NORM          = 1.0
SEED               = 42


# GRPO (PPO-style to reference)
PPO_EPS            = 0.2
KL_BETA            = 0.005
ADV_SCALE          = 1.5

# Tokenization limits
MAX_PROMPT_TOKENS     = 256 # Reduced max prompt tokens
MAX_COMPLETION_TOKENS = 256 # Reduced max completion tokens
MAX_TOTAL_TOKENS      = MAX_PROMPT_TOKENS + MAX_COMPLETION_TOKENS

# LoRA config (good coverage for LLaMA)
LORA_R         = 16
LORA_ALPHA     = 32
LORA_DROPOUT   = 0.05
LORA_TARGETS   = ["q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj"]

# Early stopping / best checkpoint
MONITOR_METRIC = "preference_accuracy"   # or "rubric_score_mean" or a composite
MONITOR_MODE   = "max"
PATIENCE       = 10
MIN_DELTA      = 1e-4
BEST_DIR       = os.path.join(OUTPUT_DIR, "best")

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
DTYPE  = torch.bfloat16 if torch.cuda.is_available() and torch.cuda.is_bf16_supported() else torch.float16

# -------------------------- UTILS --------------------------
def set_seed(seed: int = 42):
    random.seed(seed); torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)

def load_jsonl(path: str) -> List[Dict[str, Any]]:
    rows = []
    with open(path, "r", encoding="utf-8-sig") as f:
        for line in f:
            s = line.strip()
            if s: rows.append(json.loads(s))
    return rows

def completions_to_texts(comps: List[Any]) -> List[str]:
    out = []
    for c in comps:
        out.append(c["text"] if isinstance(c, dict) and "text" in c else str(c))
    return out

def totals_from_scores(row: Dict[str, Any], n: int) -> Optional[List[Optional[int]]]:
    scores = row.get("scores")
    if isinstance(scores, list) and len(scores) >= 2:
        idx2t = {s["index"]: s["total"] for s in scores if isinstance(s, dict) and "index" in s and "total" in s}
        if set(idx2t.keys()) == set(range(n)):
            return [idx2t[i] for i in range(n)]
    return None

def rank_based_adv(n: int, ranking: List[int]) -> List[float]:
    # n=4 -> base = [1.5, 0.5, -0.5, -1.5] assigned to best..worst
    base = [ (n-1)/2 - i for i in range(n) ]
    adv_by_idx = { ranking[i]: base[i] for i in range(n) }
    return [adv_by_idx[i] for i in range(n)]

# -------------------------- DATASET --------------------------
from dataclasses import dataclass
@dataclass
class Example:
    row_idx: int
    comp_idx: int
    prompt: str
    completion: str
    adv: float

class GRPODataset(Dataset):
    """Flattens prompts into (prompt, completion, advantage) examples."""
    def __init__(self, rows: List[Dict[str, Any]]):
        self.rows = rows
        exs: List[Example] = []
        for ridx, r in enumerate(rows):
            prompt = (r.get("prompt") or "").strip()
            comps  = completions_to_texts(r.get("completions", []))
            if not prompt or len(comps) < 2:
                continue
            n = len(comps)

            totals = totals_from_scores(r, n)
            if totals is not None:
                t = torch.tensor(totals, dtype=torch.float32)
                if t.std() > 0:
                    adv = ((t - t.mean()) / (t.std() + 1e-8)).tolist()
                else:
                    adv = rank_based_adv(n, r.get("ranking", list(range(n))))
            else:
                adv = rank_based_adv(n, r.get("ranking", list(range(n))))

            adv = [float(ADV_SCALE * a) for a in adv]
            for cidx, (c, a) in enumerate(zip(comps, adv)):
                exs.append(Example(row_idx=ridx, comp_idx=cidx, prompt=prompt, completion=c, adv=a))
        self.data = exs

    def __len__(self): return len(self.data)
    def __getitem__(self, idx): return self.data[idx]

# -------------------------- CUSTOM COLLATE FUNCTION --------------------------
def grpo_collate_fn(batch: List[Example]):
    """Custom collate function for the GRPODataset."""
    prompts = [ex.prompt for ex in batch]
    completions = [ex.completion for ex in batch]
    advs = torch.tensor([ex.adv for ex in batch], dtype=torch.float32)

    #  handle tokenization and padding here, similar to encode_batch
    # For now, let's just return the raw data and advantage scores
    # The tokenization and encoding will be done within the grpo_step function
    return batch, advs

# -------------------------- MODEL LOADING --------------------------
set_seed(SEED)
os.makedirs(OUTPUT_DIR, exist_ok=True); os.makedirs(BEST_DIR, exist_ok=True)

# After login(), token usually not needed; leaving out 'token/use_auth_token' for cross-version safety
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token

# Load model in 8-bit using bitsandbytes

ATTN_IMPL = "sdpa"  # or "flash_attention_2" if installed
base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID, device_map="auto", torch_dtype=DTYPE, attn_implementation=ATTN_IMPL
)
ref_model  = AutoModelForCausalLM.from_pretrained(
    MODEL_ID, device_map="auto", torch_dtype=DTYPE, attn_implementation=ATTN_IMPL
)

torch.backends.cuda.matmul.allow_tf32 = True
torch.set_float32_matmul_precision("high")


lora_cfg = LoraConfig(
    r=LORA_R, lora_alpha=LORA_ALPHA, target_modules=LORA_TARGETS,
    lora_dropout=LORA_DROPOUT, task_type="CAUSAL_LM"
)
model = get_peft_model(base_model, lora_cfg)

# Gradient checkpointing for memory
try:
    model.gradient_checkpointing_enable()
    model.enable_input_require_grads()
except Exception:
    pass

# Frozen reference model
ref_model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    device_map="auto",
    torch_dtype=DTYPE # Load in bfloat16 or float16
)
ref_model.eval()
for p in ref_model.parameters():
    p.requires_grad = False

# model.to(DEVICE); ref_model.to(DEVICE) # No need to move to device explicitly when using device_map="auto"

# -------------------------- TOKENIZATION / LOGPROBS --------------------------
def encode_pair(prompt: str, completion: str):
    p = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=MAX_PROMPT_TOKENS)
    c = tokenizer(completion, return_tensors="pt", truncation=True, max_length=MAX_COMPLETION_TOKENS)
    # avoid double BOS: drop first token of completion (usually BOS)
    input_ids = torch.cat([p["input_ids"], c["input_ids"][:, 1:]], dim=1)
    attn      = torch.cat([p["attention_mask"], torch.ones_like(c["attention_mask"][:, 1:])], dim=1)
    # truncate safety
    input_ids = input_ids[:, :MAX_TOTAL_TOKENS]
    attn      = attn[:,      :MAX_TOTAL_TOKENS]
    # completion mask aligned to target positions (shifted by 1)
    T = input_ids.size(1)
    prompt_len = p["input_ids"].shape[1]
    comp_mask_shifted = torch.zeros((1, T-1), dtype=torch.float32)
    comp_mask_shifted[:, prompt_len-1:] = 1
    return input_ids.to(DEVICE), attn.to(DEVICE), comp_mask_shifted.to(DEVICE)

def seq_logprob_from_logits(logits: torch.Tensor, input_ids: torch.Tensor, comp_mask_shifted: torch.Tensor) -> torch.Tensor:
    # shift: logits at t predict token at t+1
    logp = F.log_softmax(logits[:, :-1, :], dim=-1)           # [B, T-1, V]
    target = input_ids[:, 1:]                                  # [B, T-1]
    tok_lp = logp.gather(-1, target.unsqueeze(-1)).squeeze(-1) # [B, T-1]
    return (tok_lp * comp_mask_shifted).sum(dim=1)             # [B]

def token_kl_from_logits(logits_new, logits_ref, comp_mask_shifted):
    logp_new = F.log_softmax(logits_new[:, :-1, :], dim=-1)
    logp_ref = F.log_softmax(logits_ref[:, :-1, :], dim=-1) # Added calculation for logp_ref
    p_new = logp_new.exp()
    kl_tok = (p_new * (logp_new - logp_ref)).sum(dim=-1)      # [B, T-1]
    denom = comp_mask_shifted.sum(dim=1).clamp(min=1.0)       # [B]
    return (kl_tok * comp_mask_shifted).sum(dim=1) / denom    # [B]

def encode_batch(batch: List[Example]):
    ids, attns, masks = [], [], []
    for ex in batch:
        x, a, m = encode_pair(ex.prompt, ex.completion)
        ids.append(x); attns.append(a); masks.append(m)

    # Find the maximum sequence length in the batch
    max_len = max(id.size(1) for id in ids)

    # Pad sequences to the maximum length
    padded_ids = []
    padded_attns = []
    padded_masks = []
    for id, attn, mask in zip(ids, attns, masks):
        padding_len = max_len - id.size(1)
        padded_ids.append(F.pad(id, (0, padding_len), value=tokenizer.pad_token_id))
        padded_attns.append(F.pad(attn, (0, padding_len), value=0))
        # Pad the completion mask as well, ensuring it's still T-1
        padded_masks.append(F.pad(mask, (0, padding_len), value=0))

    input_ids = torch.vstack(padded_ids); attention_mask = torch.vstack(padded_attns); comp_mask_shifted = torch.vstack(padded_masks)
    return input_ids.to(DEVICE), attention_mask.to(DEVICE), comp_mask_shifted.to(DEVICE)

# -------------------------- GRPO STEP --------------------------
def grpo_step(model, ref_model, batch_data: List[Example], advs: torch.Tensor) -> torch.Tensor:
    input_ids, attn, comp_mask_shifted = encode_batch(batch_data)
    with torch.cuda.amp.autocast(dtype=torch.float16, enabled=True): # Use float16 for mixed precision
        logits_m = model(input_ids=input_ids, attention_mask=attn).logits
        lp_new   = seq_logprob_from_logits(logits_m, input_ids, comp_mask_shifted)  # [B]
    with torch.no_grad():
        with torch.cuda.amp.autocast(dtype=torch.float16, enabled=True): # Use float16 for mixed precision
            logits_r = ref_model(input_ids=input_ids, attention_mask=attn).logits
            lp_ref   = seq_logprob_from_logits(logits_r, input_ids, comp_mask_shifted)  # [B]
    delta = (lp_new - lp_ref).clamp(-20, 20)                # [B]
    ratio = torch.exp(delta)
    # adv   = torch.tensor([ex.adv for ex in batch_data], dtype=torch.float32, device=DEVICE) # Moved to collate_fn
    unclipped = ratio * advs.to(DEVICE)
    clipped   = torch.clamp(ratio, 1 - PPO_EPS, 1 + PPO_EPS) * advs.to(DEVICE)
    policy_obj = torch.min(unclipped, clipped)
    loss = -(policy_obj - KL_BETA * delta).mean()
    return loss

# -------------------------- METRICS --------------------------
def eval_metrics(policy, ref_policy, rows: List[Dict[str,Any]], batch_size: int = 8):
    policy.eval();  ref_policy.eval()
    # Flatten for batched scoring
    flat, texts_cache, totals_cache, prefs_cache = [], {}, {}, {}
    for ridx, r in enumerate(rows):
        comps = completions_to_texts(r.get("completions", []))
        if len(comps) < 2: continue
        texts_cache[ridx]  = comps
        totals_cache[ridx] = totals_from_scores(r, len(comps))
        prefs_cache[ridx]  = r.get("preferences", [])
        for cidx, c in enumerate(comps):
            flat.append(Example(row_idx=ridx, comp_idx=cidx, prompt=r.get("prompt",""), completion=c, adv=0.0))

    lp_model, lp_ref, kl_items = {}, {}, []
    for i in tqdm(range(0, len(flat), batch_size), desc="Eval scoring", leave=False):
        batch = flat[i:i+batch_size]
        # Use encode_batch here since we are not using a DataLoader for eval
        input_ids, attn, comp_mask_shifted = encode_batch(batch)
        with torch.no_grad(), torch.cuda.amp.autocast(dtype=torch.float16, enabled=True): # Use float16 for mixed precision
            logits_m = policy(input_ids=input_ids, attention_mask=attn).logits
            logits_r = ref_policy(input_ids=input_ids, attention_mask=attn).logits
            lpm = seq_logprob_from_logits(logits_m, input_ids, comp_mask_shifted)
            lpr = seq_logprob_from_logits(logits_r, input_ids, comp_mask_shifted)
            kl  = token_kl_from_logits(logits_m, logits_r, comp_mask_shifted)
        for j, ex in enumerate(batch):
            lp_model[(ex.row_idx, ex.comp_idx)] = float(lpm[j].detach().cpu())
            lp_ref[(ex.row_idx, ex.comp_idx)]   = float(lpr[j].detach().cpu())
            kl_items.append(float(kl[j].detach().cpu()))

    # Aggregate per-prompt metrics
    pair_total = pair_correct = pair_correct_ref = model_wins_vs_ref = 0
    margins, top_scores = [], []
    for ridx, comps in texts_cache.items():
        n = len(comps)
        ms = [lp_model[(ridx, i)] for i in range(n)]
        rs = [lp_ref[(ridx, i)]   for i in range(n)]
        model_order = sorted(range(n), key=lambda i: ms[i], reverse=True)

        totals = totals_cache.get(ridx)
        if totals and any(t is not None for t in totals):
            top_scores.append(totals[model_order[0]])

        for p in prefs_cache.get(ridx, []):
            w, l = p["winner"], p["loser"]
            pair_total += 1
            model_ok = (ms[w] > ms[l])
            ref_ok   = (rs[w] > rs[l])
            if model_ok:
                pair_correct += 1
                margins.append(ms[w] - ms[l])
            if ref_ok:
                pair_correct_ref += 1
            if model_ok and not ref_ok:
                model_wins_vs_ref += 1

    pref_acc  = pair_correct / max(1, pair_total)
    ref_acc   = pair_correct_ref / max(1, pair_total)
    win_rate  = model_wins_vs_ref / max(1, pair_total)
    kl_mean   = sum(kl_items) / max(1, len(kl_items))
    margin_mu = (sum(margins)/len(margins)) if margins else 0.0
    margin_sd = float(torch.tensor(margins).std().item()) if margins else 0.0
    rub_mu    = (sum(top_scores)/len(top_scores)) if top_scores else None
    _valid = [float(t) for t in top_scores if t is not None]
    rub_sd = float(torch.tensor(_valid, dtype=torch.float32).std().item()) if _valid else 0.0


    return {
        "rubric_score_mean": rub_mu,
        "rubric_score_std": rub_sd,
        "preference_accuracy": pref_acc,
        "preference_accuracy_ref": ref_acc,
        "win_rate_vs_ref": win_rate,
        "kl_divergence_mean": kl_mean,
        "margin_of_victory_mean": margin_mu,
        "margin_of_victory_std":  margin_sd,
        "pairs_count": pair_total,
        "examples_scored": len(flat),
        "prompts_count": len(texts_cache),
    }

def monitored_value(metrics: dict) -> float:
    return float(metrics.get(MONITOR_METRIC, 0.0) or 0.0)

# -------------------------- LOAD & SPLIT DATA --------------------------
rows_all = load_jsonl(DATA_JSONL)
random.seed(SEED)
perm = list(range(len(rows_all))); random.shuffle(perm)
num_val = max(1, int(len(rows_all) * VAL_FRACTION))
val_ids = set(perm[:num_val])
train_rows = [rows_all[i] for i in range(len(rows_all)) if i not in val_ids]
val_rows   = [rows_all[i] for i in range(len(rows_all)) if i in val_ids]

train_ds = GRPODataset(train_rows)
val_ds   = GRPODataset(val_rows)


optim = torch.optim.AdamW(model.parameters(), lr=LR, betas=(0.9,0.95),
                          weight_decay=WEIGHT_DECAY, fused=True)
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, drop_last=True,
                          num_workers=4, pin_memory=True, persistent_workers=True, prefetch_factor=4,
                          collate_fn=grpo_collate_fn) # Changed collate_fn to grpo_collate_fn

# -------------------------- OPTIM & SCHED --------------------------

if MAX_STEPS is None:
    steps_per_epoch = max(1, math.floor(len(train_loader) / GRAD_ACCUM_STEPS))
    MAX_STEPS = steps_per_epoch * NUM_EPOCHS
sched  = get_cosine_schedule_with_warmup(optim, num_warmup_steps=WARMUP_STEPS, num_training_steps=MAX_STEPS)
scaler = torch.cuda.amp.GradScaler(enabled=True)

# -------------------------- TRAIN w/ EARLY STOP + TQDM --------------------------
best_score = None
patience_left = PATIENCE
global_step = 0

for epoch in trange(NUM_EPOCHS, desc="Epochs"):
    model.train()
    running = 0.0
    accum = 0
    step_bar = tqdm(train_loader, desc=f"Train E{epoch+1}", leave=False)
    for batch_data, advs in step_bar:
        loss = grpo_step(model, ref_model, batch_data, advs)
        scaler.scale(loss / GRAD_ACCUM_STEPS).backward()
        accum += 1

        if accum % GRAD_ACCUM_STEPS == 0:
            torch.nn.utils.clip_grad_norm_(model.parameters(), CLIP_NORM)
            scaler.step(optim); scaler.update()
            optim.zero_grad(set_to_none=True)
            sched.step()
            global_step += 1
            running += float(loss.detach().cpu())
            step_bar.set_postfix({
                "loss": f"{running/max(1, global_step%10 or 1):.4f}",
                "lr": f"{sched.get_last_lr()[0]:.2e}",
                "gs": global_step
            })
    step_bar.close()

    # ---- EVAL METRICS ----
    with torch.no_grad():
        metrics = eval_metrics(model, ref_model, val_rows, batch_size=max(4, BATCH_SIZE))
    cur = monitored_value(metrics)
    print(f"Epoch {epoch+1} metrics:\n{json.dumps(metrics, indent=2)}")

    # Save epoch metrics & checkpoint
    os.makedirs(OUTPUT_DIR, exist_ok=True)
    with open(os.path.join(OUTPUT_DIR, f"metrics_epoch{epoch+1}.json"), "w", encoding="utf-8") as f:
        json.dump(metrics, f, indent=2)
    epoch_dir = os.path.join(OUTPUT_DIR, f"lora_epoch{epoch+1}")
    model.save_pretrained(epoch_dir); tokenizer.save_pretrained(epoch_dir)
    print(f"saved epoch checkpoint → {epoch_dir}")

    # ---- EARLY STOP / BEST SAVE ----
    improved = (best_score is None)
    if best_score is not None:
        if MONITOR_MODE == "max":
            improved = (cur - best_score) > MIN_DELTA
        else:
            improved = (best_score - cur) > MIN_DELTA

    if improved:
        best_score = cur
        patience_left = PATIENCE
        # overwrite BEST_DIR
        for f in os.listdir(BEST_DIR):
            try: os.remove(os.path.join(BEST_DIR, f))
            except: pass
        model.save_pretrained(BEST_DIR); tokenizer.save_pretrained(BEST_DIR)
        with open(os.path.join(BEST_DIR, "best_metrics.json"), "w", encoding="utf-8") as f:
            json.dump({"epoch": epoch+1, "score": best_score, "monitor": MONITOR_METRIC,
                       "mode": MONITOR_MODE, **metrics}, f, indent=2)
        print(f"new BEST ({MONITOR_METRIC}={best_score:.6f}) → saved to {BEST_DIR}")
    else:
        patience_left -= 1
        print(f"no improvement on '{MONITOR_METRIC}' (best={best_score:.6f}, cur={cur:.6f}); patience left: {patience_left}")
        if patience_left <= 0:
            print("Early stopping triggered.")
            break

print("Training complete.")
print(f"Best checkpoint directory: {BEST_DIR}")

/tmp/ipython-input-3046456154.py:406: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler(enabled=True)


Epochs:   0%|          | 0/40 [00:00<?, ?it/s]

Train E1:   0%|          | 0/44 [00:00<?, ?it/s]

/tmp/ipython-input-3046456154.py:281: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(dtype=torch.float16, enabled=True): # Use float16 for mixed precision
/tmp/ipython-input-3046456154.py:285: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(dtype=torch.float16, enabled=True): # Use float16 for mixed precision


Eval scoring:   0%|          | 0/22 [00:00<?, ?it/s]

...

Eval scoring:   0%|          | 0/22 [00:00<?, ?it/s]

📊 Epoch 12 metrics:
{
  "rubric_score_mean": 9.636363636363637,
  "rubric_score_std": 1.2552918195724487,
  "preference_accuracy": 0.5454545454545454,
  "preference_accuracy_ref": 0.5757575757575758,
  "win_rate_vs_ref": 0.08333333333333333,
  "kl_divergence_mean": 1.4609511934898116,
  "margin_of_victory_mean": 138.25139066908093,
  "margin_of_victory_std": 127.32368469238281,
  "pairs_count": 132,
  "examples_scored": 88,
  "prompts_count": 22
}
💾 saved epoch checkpoint → /content/drive/MyDrive/grpo_lora_ckpt/lora_epoch12
⏳ no improvement on 'preference_accuracy' (best=0.575758, cur=0.545455); patience left: 1


Train E13:   0%|          | 0/44 [00:00<?, ?it/s]

Eval scoring:   0%|          | 0/22 [00:00<?, ?it/s]

📊 Epoch 13 metrics:
{
  "rubric_score_mean": 9.636363636363637,
  "rubric_score_std": 1.2552918195724487,
  "preference_accuracy": 0.5454545454545454,
  "preference_accuracy_ref": 0.5757575757575758,
  "win_rate_vs_ref": 0.08333333333333333,
  "kl_divergence_mean": 1.466544004326517,
  "margin_of_victory_mean": 138.36251915825738,
  "margin_of_victory_std": 127.31489562988281,
  "pairs_count": 132,
  "examples_scored": 88,
  "prompts_count": 22
}
💾 saved epoch checkpoint → /content/drive/MyDrive/grpo_lora_ckpt/lora_epoch13
⏳ no improvement on 'preference_accuracy' (best=0.575758, cur=0.545455); patience left: 0
🛑 Early stopping triggered.
Training complete.
Best checkpoint directory: /content/drive/MyDrive/grpo_lora_ckpt/best


In [8]:
!pip show transformers

Name: transformers
Version: 4.55.1
Summary: State-of-the-art Machine Learning for JAX, PyTorch and TensorFlow
Home-page: https://github.com/huggingface/transformers
Author: The Hugging Face team (past and future) with the help of all our contributors (https://github.com/huggingface/transformers/graphs/contributors)
Author-email: transformers@huggingface.co
License: Apache 2.0 License
Location: /usr/local/lib/python3.11/dist-packages
Requires: filelock, huggingface-hub, numpy, packaging, pyyaml, regex, requests, safetensors, tokenizers, tqdm
Required-by: peft, sentence-transformers


In [9]:
!pip install -U transformers

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.0/42.0 kB 1.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.3/11.3 MB 117.3 MB/s eta 0:00:00
  Attempting uninstall: transformers
    Found existing installation: transformers 4.55.1
    Uninstalling transformers-4.55.1:
      Successfully uninstalled transformers-4.55.1


Let's start by inspecting the model's memory usage and the size of the tensors being created during a forward pass.

In [ ]:
# Print model summary to see parameter counts and estimated memory usage


# Get a sample batch from the DataLoader
sample_batch_data, sample_advs = next(iter(train_loader))

# Encode the sample batch to see the tensor sizes
input_ids, attn, comp_mask_shifted = encode_batch(sample_batch_data)

print("\nTensor sizes in a sample batch:")
print(f"input_ids: {input_ids.size()}")
print(f"attention_mask: {attn.size()}")
print(f"comp_mask_shifted: {comp_mask_shifted.size()}")

#  check the memory usage of these tensors
print(f"input_ids memory (bytes): {input_ids.element_size() * input_ids.nelement()}")
print(f"attention_mask memory (bytes): {attn.element_size() * attn.nelement()}")
print(f"comp_mask_shifted memory (bytes): {comp_mask_shifted.element_size() * comp_mask_shifted.nelement()}")

# To get a more detailed breakdown of GPU memory usage during runtime,

# or integrate profiling tools within the training loop.


Tensor sizes in a sample batch:
input_ids: torch.Size([2, 304])
attention_mask: torch.Size([2, 304])
comp_mask_shifted: torch.Size([2, 303])
input_ids memory (bytes): 4864
attention_mask memory (bytes): 4864
comp_mask_shifted memory (bytes): 2424


In [11]:
!pip install bitsandbytes

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.3/61.3 MB 39.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 110.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 0.0/24.6 MB ? eta -:--:--^C


In [ ]:
!pip install bitsandbytes

^C


In [ ]:
!pip install -U "transformers>=4.43,<5" "peft>=0.11.1,<0.13" "accelerate>=0.30.0" "safetensors>=0.4.2" "bitsandbytes>=0.43.0" tqdm